# Study 962 — Do It Yourself — the teardown

The depth × weighting sweep, the HAC *t* on the daily gap, block-bootstrap CIs, the power arithmetic, the era cut, the cost and rebalance-frequency sweeps, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `1f3a3deb14a7`), sample 2011-01-03 → 2026-06-30, 3,895 days, monthly rebalance, one-day execution lag, 5 bps one-way × NAV, long-only (no borrow leg).

In [1]:
R = {'start': '2011-01-03', 'end': '2026-06-30', 'n_days': 3895, 'fp': '1f3a3deb14a7', 'n_tickers': 48, 'fee': 0.08, 'blend_gap': -0.76, 'blend_t': -0.58, 'blend_te': 5.13, 'blend_ci_lo': -3.21, 'blend_ci_hi': 1.69, 'blend_worst12': -12.3, 'look_gap': 6.08, 'look_t': 6.1, 'look_te': 3.85, 'look_ci_lo': 4.07, 'look_ci_hi': 8.11, 'look_worst12': -8.1, 'raw_premium': 6.84, 'eq2026_gap': 5.06, 'eq2026_t': 5.13, 'vintage_premium': 5.81, 'weighting_premium': 1.02, 'decomp': {'XLK': (-3.94, 4.16, 9.83), 'XLE': (-0.3, 5.56, 3.35), 'XLF': (2.07, 5.46, 5.17)}, 'xlk_gap': -3.94, 'xlk_t': -1.86, 'xlk_te': 9.13, 'xlk_ci_lo': -7.77, 'xlk_ci_hi': 0.1, 'xlk_corr': 0.915, 'xlk_beta': 0.76, 'xlk_sh_diy': 0.86, 'xlk_sh_fund': 0.9, 'xlk_dd_diy': -29.9, 'xlk_dd_fund': -33.6, 'xlk_worst12': -23.3, 'xle_gap': -0.3, 'xle_t': -0.12, 'xle_te': 9.75, 'xle_ci_lo': -5.24, 'xle_ci_hi': 4.58, 'xle_corr': 0.969, 'xle_beta': 1.19, 'xle_sh_diy': 0.25, 'xle_sh_fund': 0.32, 'xle_dd_diy': -81.8, 'xle_dd_fund': -71.3, 'xle_worst12': -25.7, 'xlf_gap': 2.07, 'xlf_t': 1.32, 'xlf_te': 6.25, 'xlf_ci_lo': -0.94, 'xlf_ci_hi': 5.14, 'xlf_corr': 0.976, 'xlf_beta': 1.14, 'xlf_sh_diy': 0.55, 'xlf_sh_fund': 0.55, 'xlf_dd_diy': -47.3, 'xlf_dd_fund': -42.9, 'xlf_worst12': -11.1, 'sweep': {('XLK', 'hindsight'): [(13.62, 3.66), (12.93, 4.16), (9.83, 3.89)], ('XLK', 'contemp'): [(-1.75, -0.72), (-2.9, -1.19), (-3.94, -1.86)], ('XLE', 'hindsight'): [(1.46, 0.72), (2.46, 1.57), (3.35, 2.82)], ('XLE', 'contemp'): [(-0.74, -0.42), (-0.12, -0.07), (-0.3, -0.12)], ('XLF', 'hindsight'): [(4.62, 2.52), (5.47, 3.2), (5.17, 4.02)], ('XLF', 'contemp'): [(2.25, 1.32), (2.17, 1.13), (2.07, 1.32)]}, 'se_ann': 1.31, 'se_over_fee': 16, 'years_needed': 16472, 'sample_years': 15.4, 'era': {'XLK': ((-1.35, -0.82, 5.02), (-6.94, -1.73, 12.04)), 'XLE': ((-1.42, -0.74, 5.72), (0.8, 0.17, 12.72)), 'XLF': ((-0.25, -0.17, 4.38), (4.46, 1.58, 7.76))}, 'cost': {0.0: (-3.92, -1.85), 5.0: (-3.94, -1.86), 25.0: (-4.04, -1.91), 50.0: (-4.17, -1.97)}, 'freq': {'M': (-3.94, -1.86, 9.13, 0.52), 'Q': (-3.65, -1.73, 9.01, 0.31), 'A': (-3.66, -1.75, 8.94, 0.14), 'N': (-3.79, -2.38, 7.08, 0.0)}, 'cal_years': [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], 'cal_xlk': [6.9, -0.9, -2.8, -3.7, -4.2, 3.3, -12.8, 1.7, -10.2, -24.0, -7.6, 6.3, -20.5, -4.0, 8.6, -10.0], 'cal_xle': [-4.2, -4.3, 0.9, 2.2, -2.9, 5.1, -2.7, -6.6, -8.2, 3.8, 12.2, 10.8, -6.2, -20.2, -2.7, 0.4], 'cal_xlf': [-5.1, 5.3, 5.4, -1.6, -2.8, 0.2, -1.1, -2.0, 4.3, -6.8, 0.5, 0.5, 2.9, 7.7, 14.0, 8.2], 'syn_fee': 2.0, 'syn_pl_gap': 1.957, 'syn_pl_sd': 0.131, 'syn_pl_t': 12.52, 'syn_pl_fire': 6, 'syn_nl_gap': -0.062, 'syn_nl_sd': 0.128, 'syn_nl_t': -0.39, 'syn_nl_fire': 0, 'syn_te': 0.69}

## 1. The identification problem

A top-N basket built from **today's** holdings list is a survivorship portfolio: the 2011 investor could not have known it. The study's whole design is the contrast between that basket and one built from the **January-2011** list, held equal weight and fixed from the first day of the sample.

> 💡 **In plain words:** one basket is allowed to peek at the answer sheet; the other is not.

In [2]:
print('depth x weighting sweep — annualised gap (HAC t), monthly, 5 bps')
print('%-6s %-12s %14s %14s %14s' % ('fund','scheme','N=3','N=5','N=10'))
for fund in ('XLK','XLE','XLF'):
    for scheme in ('hindsight','contemp'):
        cells = R['sweep'][(fund, scheme)]
        print('%-6s %-12s ' % (fund, scheme) +
              ' '.join('%+8.2f%% (%+.2f)' % (g, t) for g, t in cells))
hits_look = sum(abs(t) >= 2 for k, v in R['sweep'].items() if k[1]=='hindsight' for _, t in v)
hits_cont = sum(abs(t) >= 2 for k, v in R['sweep'].items() if k[1]=='contemp' for _, t in v)
print('\n|t| >= 2 cells: hindsight %d/9, contemporaneous %d/9' % (hits_look, hits_cont))

depth x weighting sweep — annualised gap (HAC t), monthly, 5 bps
fund   scheme                  N=3            N=5           N=10
XLK    hindsight      +13.62% (+3.66)   +12.93% (+4.16)    +9.83% (+3.89)
XLK    contemp         -1.75% (-0.72)    -2.90% (-1.19)    -3.94% (-1.86)
XLE    hindsight       +1.46% (+0.72)    +2.46% (+1.57)    +3.35% (+2.82)
XLE    contemp         -0.74% (-0.42)    -0.12% (-0.07)    -0.30% (-0.12)
XLF    hindsight       +4.62% (+2.52)    +5.47% (+3.20)    +5.17% (+4.02)
XLF    contemp         +2.25% (+1.32)    +2.17% (+1.13)    +2.07% (+1.32)

|t| >= 2 cells: hindsight 7/9, contemporaneous 0/9


## 2. The blended headline, and what the raw premium actually contains

One number per scheme: the equal-weight blend of the three sectors' top-10 gap series, with a 21-day circular block-bootstrap CI on the annualised mean.

The naive move here is to call the whole hindsight-minus-contemporaneous difference the look-ahead. It is not: `cap2026` and `eq2011` differ in **two** factors at once — the *membership vintage* and the *weighting scheme*. The `eq2026` basket (today's names, **equal** weight) is the control that identifies them separately.

> 💡 **In plain words:** two knobs were turned, so measure them one at a time.

In [3]:
print('%-22s %10s %8s %8s %20s %10s' % ('basket','ann gap','HAC t','TE','95% CI','worst 12m'))
print('%-22s %+9.2f%% %+8.2f %7.2f%% [%+6.2f%%, %+6.2f%%] %9.1f%%'
      % ('contemporaneous 2011', R['blend_gap'], R['blend_t'], R['blend_te'],
         R['blend_ci_lo'], R['blend_ci_hi'], R['blend_worst12']))
print('%-22s %+9.2f%% %+8.2f %7.2f%% [%+6.2f%%, %+6.2f%%] %9.1f%%'
      % ('hindsight 2026', R['look_gap'], R['look_t'], R['look_te'],
         R['look_ci_lo'], R['look_ci_hi'], R['look_worst12']))
print('%-22s %+9.2f%% %+8.2f' % ('eq2026 control', R['eq2026_gap'], R['eq2026_t']))

print('\ndecomposition of the %+.2f%%/yr raw difference' % R['raw_premium'])
print('  name vintage (eq2026 - eq2011, weighting fixed): %+.2f%%/yr  <- the look-ahead'
      % R['vintage_premium'])
print('  weighting    (cap2026 - eq2026, names fixed)   : %+.2f%%/yr  <- not hindsight'
      % R['weighting_premium'])
print('\nper sector: eq2011 -> eq2026 -> cap2026   (vintage | weighting)')
for f, (a, b, c) in R['decomp'].items():
    print('  %-4s %+6.2f%% -> %+6.2f%% -> %+6.2f%%   (%+6.2f | %+6.2f)'
          % (f, a, b, c, b - a, c - b))
print('\nvintage term is positive in 3/3 sectors; the weighting term is not.')

basket                    ann gap    HAC t       TE               95% CI  worst 12m
contemporaneous 2011       -0.76%    -0.58    5.13% [ -3.21%,  +1.69%]     -12.3%
hindsight 2026             +6.08%    +6.10    3.85% [ +4.07%,  +8.11%]      -8.1%
eq2026 control             +5.06%    +5.13

decomposition of the +6.84%/yr raw difference
  name vintage (eq2026 - eq2011, weighting fixed): +5.81%/yr  <- the look-ahead
  weighting    (cap2026 - eq2026, names fixed)   : +1.02%/yr  <- not hindsight

per sector: eq2011 -> eq2026 -> cap2026   (vintage | weighting)
  XLK   -3.94% ->  +4.16% ->  +9.83%   ( +8.10 |  +5.67)
  XLE   -0.30% ->  +5.56% ->  +3.35%   ( +5.86 |  -2.21)
  XLF   +2.07% ->  +5.46% ->  +5.17%   ( +3.39 |  -0.29)

vintage term is positive in 3/3 sectors; the weighting term is not.


## 3. Per sector — the excess-of-cash race and the risk you inherit

Both legs are measured excess of BIL's actual total return, so the cash leg cancels in the difference and the Sharpe comparison is like for like.

In [4]:
rows = [('XLK', R['xlk_gap'], R['xlk_t'], R['xlk_te'], R['xlk_corr'], R['xlk_beta'],
         R['xlk_sh_diy'], R['xlk_sh_fund'], R['xlk_dd_diy'], R['xlk_dd_fund'], R['xlk_worst12']),
        ('XLE', R['xle_gap'], R['xle_t'], R['xle_te'], R['xle_corr'], R['xle_beta'],
         R['xle_sh_diy'], R['xle_sh_fund'], R['xle_dd_diy'], R['xle_dd_fund'], R['xle_worst12']),
        ('XLF', R['xlf_gap'], R['xlf_t'], R['xlf_te'], R['xlf_corr'], R['xlf_beta'],
         R['xlf_sh_diy'], R['xlf_sh_fund'], R['xlf_dd_diy'], R['xlf_dd_fund'], R['xlf_worst12'])]
print('%-5s %8s %7s %7s %6s %6s %14s %16s %10s'
      % ('fund','gap','HAC t','TE','corr','beta','exSharpe D/F','maxDD D/F','worst12m'))
for f, g, t, te, c, b, sd, sf, dd, df, w in rows:
    print('%-5s %+7.2f%% %+7.2f %6.2f%% %6.3f %6.2f  %+5.2f/%+5.2f (%+.2f) %6.1f%%/%6.1f%% %9.1f%%'
          % (f, g, t, te, c, b, sd, sf, sd - sf, dd, df, w))

fund       gap   HAC t      TE   corr   beta   exSharpe D/F        maxDD D/F   worst12m
XLK     -3.94%   -1.86   9.13%  0.915   0.76  +0.86/+0.90 (-0.04)  -29.9%/ -33.6%     -23.3%
XLE     -0.30%   -0.12   9.75%  0.969   1.19  +0.25/+0.32 (-0.07)  -81.8%/ -71.3%     -25.7%
XLF     +2.07%   +1.32   6.25%  0.976   1.14  +0.55/+0.55 (+0.00)  -47.3%/ -42.9%     -11.1%


## 4. Power — the test cannot see an 8 bp fee

This is the finding, not a caveat. The replication noise is three orders of magnitude larger than the quantity being estimated.

> 💡 **In plain words:** you are trying to hear a whisper next to a jet engine.

In [5]:
te, fee, yrs = R['blend_te'], R['fee'], R['sample_years']
se = te / yrs**0.5
print('blended TE            : %.2f%%/yr' % te)
print('sample length         : %.1f yr' % yrs)
print('SE of annualised gap  : %.2f%%/yr  (= %.0fx the %.2f%% fee)' % (se, se/fee, fee))
print('years needed for |t|=2: %s' % format(int(round((2*te/fee)**2)), ','))
print('\nfrozen values: SE %.2f%%, ratio %dx, years %s'
      % (R['se_ann'], R['se_over_fee'], format(R['years_needed'], ',')))

blended TE            : 5.13%/yr
sample length         : 15.4 yr
SE of annualised gap  : 1.31%/yr  (= 16x the 0.08% fee)
years needed for |t|=2: 16,448

frozen values: SE 1.31%, ratio 16x, years 16,472


## 5. Era cut (split 2019-01-01)

No sector holds its sign, and the tracking error roughly doubles in the recent era: the funds have grown more top-heavy, so a list fixed in 2011 drifts further from the thing it is replacing.

In [6]:
print('%-5s %26s %26s' % ('fund','2011-2018  gap (t) / TE','2019-2026  gap (t) / TE'))
for f, ((g1,t1,e1),(g2,t2,e2)) in R['era'].items():
    print('%-5s %+14.2f%% (%+.2f) %5.2f%% %+14.2f%% (%+.2f) %5.2f%%'
          % (f, g1, t1, e1, g2, t2, e2))

fund     2011-2018  gap (t) / TE    2019-2026  gap (t) / TE
XLK            -1.35% (-0.82)  5.02%          -6.94% (-1.73) 12.04%
XLE            -1.42% (-0.74)  5.72%          +0.80% (+0.17) 12.72%
XLF            -0.25% (-0.17)  4.38%          +4.46% (+1.58)  7.76%


## 6. Cost and rebalance sweeps (XLK, contemporaneous top-10)

Turnover is ~0.5×/yr, so friction decides nothing — gross and net differ by 2 bps at the house rate. Long-only, so there is no borrow rate to sweep.

Across the **full** contemporaneous grid — 3 funds × 3 depths × 4 frequencies = **36 cells** — exactly one clears |*t*| = 2: the never-rebalanced XLK top-10, at −2.38, *losing* to its fund. We do not bank that either. Thirty-six correlated cells at 5% size would be expected to throw up **≈1.8** hits under a true null, so one hit is if anything fewer than noise predicts.

> 💡 **In plain words:** we searched hard for a winner and did not even find enough losers to be surprising.

In [7]:
print('one-way cost sweep')
for c in sorted(R['cost']):
    g, t = R['cost'][c]
    print('  %5.1f bps: gap %+.2f%% (t %+.2f)' % (c, g, t))
print('\nrebalance-frequency sweep')
names = {'M':'monthly','Q':'quarterly','A':'annual','N':'never'}
for k in ('M','Q','A','N'):
    g, t, te, tu = R['freq'][k]
    flag = '  <-- |t| >= 2, wrong sign' if abs(t) >= 2 else ''
    print('  %-10s gap %+.2f%% (t %+.2f)  TE %5.2f%%  turnover %.2f/yr%s'
          % (names[k], g, t, te, tu, flag))

one-way cost sweep
    0.0 bps: gap -3.92% (t -1.85)
    5.0 bps: gap -3.94% (t -1.86)
   25.0 bps: gap -4.04% (t -1.91)
   50.0 bps: gap -4.17% (t -1.97)

rebalance-frequency sweep
  monthly    gap -3.94% (t -1.86)  TE  9.13%  turnover 0.52/yr
  quarterly  gap -3.65% (t -1.73)  TE  9.01%  turnover 0.31/yr
  annual     gap -3.66% (t -1.75)  TE  8.94%  turnover 0.14/yr
  never      gap -3.79% (t -2.38)  TE  7.08%  turnover 0.00/yr  <-- |t| >= 2, wrong sign


## 7. Live synthetic control — the detector is unbiased

A one-factor sector world: ten big names, a forty-name tail the basket omits, and a fund that charges a **deliberately fat** 2%/yr against a **deliberately thin** 10% tail — the only regime in which a test of this shape has power. The detector must recover the planted fee and stay silent when it is switched off.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from diy_sector import data, strategy as st
for ss, tag in [(1.0, 'planted fee'), (0.0, 'null (no fee)')]:
    gaps, ts = [], []
    for s in range(4):
        p, t = data.synthetic_panel(n_years=10, signal_strength=ss, seed=962+s)
        d = st.synthetic_detect(p, t, cost_bps=0.0)
        gaps.append(d['ann_gap']); ts.append(d['t_diff'])
    gaps, ts = np.array(gaps), np.array(ts)
    print('%-14s planted %.2f%%/yr -> gap %+.3f%% (sd %.3f%%), mean t %+.2f, |t|>=2 in %d/4'
          % (tag, t['planted_fee_ann']*100, gaps.mean()*100, gaps.std(ddof=1)*100,
             ts.mean(), (np.abs(ts) >= 2).sum()))

planted fee    planted 2.00%/yr -> gap +1.920% (sd 0.277%), mean t +8.88, |t|>=2 in 4/4


null (no fee)  planted 0.00%/yr -> gap -0.098% (sd 0.272%), mean t -0.47, |t|>=2 in 0/4


## Verdict

- **Signal — None.** Contemporaneous blend **-0.76%/yr**, HAC *t* = **-0.58**, bootstrap CI [-3.21%, +1.69%] straddling zero; 0/9 contemporaneous cells clear |*t*| = 2 against 7/9 hindsight cells; no sign survives the era cut. Decomposed against the `eq2026` control, the look-ahead premium is **+5.81%/yr** (positive in 3/3 sectors) and the remaining +1.02%/yr of the raw +6.84% is cap- versus equal-weighting, which is not hindsight at all — it is +5.67 pp in XLK and *negative* in XLE and XLF. Survivorship is named on this axis twice: the hindsight list *is* a survivor list, and the 2011 control is survivor-conditional throughout (in energy explicitly — two acquired constituents have no retrievable tape), a bias that runs in the do-it-yourself basket's favour and still fails to produce an edge. The synthetic control recovers **+1.96%** of a planted **2.00%** fee (*t* = +12.52, 6/6 seeds) and returns **-0.062%** on the null (0/6), so the harness is sound.
- **Tradability — Mirage.** Fee saved **0.08%/yr**; tracking error accepted **5.13%/yr** blended and 9.1–9.8%/yr per sector — **64×** to **122×** the prize. Worst rolling twelve months -12.3% blended, -25.7% in energy; energy drawdown -81.8% vs the fund's -71.3%. None of it is compensated: excess-of-cash Sharpe advantage -0.04 / -0.07 / +0.00.
- **What it really is.** A concentrated, fixed-list active sector bet wearing an indexing costume. The fee is the smallest term in the problem by two orders of magnitude.